# Customer Churn Prediction & Retention Analytics

## Business Problem

Customer churn reduces recurring revenue and increases acquisition cost. The goal of this notebook is to build an interpretable churn prediction workflow that identifies customers at risk and translates model outputs into retention actions.

This project keeps the original Telco preprocessing approach, then modernizes the model training, evaluation, tuning, explainability, and business recommendation layers.


## Data Understanding

The dataset contains customer demographics, subscribed services, contract details, billing method, tenure, monthly charges, total charges, and the target variable `Churn`.

For churn use cases, accuracy alone is not enough because the business usually cares more about identifying churners early. We will compare models using ROC-AUC, recall, precision, F1-score, confusion matrices, and classification reports.


In [ ]:
from pathlib import Path
import sys
import pickle
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid", palette="Set2")

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from imblearn.combine import SMOTEENN
from scipy.stats import randint, uniform
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import ConfusionMatrixDisplay
from sklearn.model_selection import RandomizedSearchCV, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier
from xgboost import XGBClassifier

from src.evaluate import evaluate_classifier, metrics_table
from src.preprocessing import load_raw_data, make_model_frame, split_features_target


In [ ]:
RANDOM_STATE = 42
DATA_PATH = PROJECT_ROOT / "data" / "raw" / "WA_Fn-UseC_-Telco-Customer-Churn.csv"
MODEL_PATH = PROJECT_ROOT / "models" / "best_model.pkl"

raw_df = load_raw_data(DATA_PATH)
raw_df.head()


In [ ]:
raw_df.shape, raw_df.dtypes


In [ ]:
raw_df["Churn"].value_counts(normalize=True).mul(100).round(2)


## Data Cleaning

We preserve the original project logic because it is reasonable for this dataset:

- Convert `TotalCharges` from object to numeric.
- Drop the 11 rows where `TotalCharges` is missing after conversion. This is only about 0.15% of the dataset.
- Convert `Churn` from `Yes`/`No` to `1`/`0`.


## Feature Engineering

The original notebook grouped customer tenure into yearly buckets. This keeps the model easy to explain in interviews and aligns with churn behavior: new customers often churn differently from long-tenure customers.

We keep that feature, drop raw `tenure`, drop `customerID`, and one-hot encode categorical columns.


In [ ]:
model_df = make_model_frame(raw_df)
model_df.head()


In [ ]:
X, y = split_features_target(model_df)
print(f"Feature matrix: {X.shape}")
print(f"Target distribution:
{y.value_counts(normalize=True).round(3)}")


## Data Preprocessing

The train/test split is stratified so the churn ratio is preserved in both sets. This gives a more reliable estimate of real-world performance.


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=y,
)

print(X_train.shape, X_test.shape)
y_train.value_counts(normalize=True).round(3)


## Handling Imbalance

The dataset is imbalanced: non-churn customers are the majority class. We keep the original project's SMOTE-style approach, but apply it only to the training data to avoid test-set leakage.

`SMOTEENN` first creates synthetic minority samples and then cleans ambiguous samples using Edited Nearest Neighbors. This can help recall, which matters when retention teams want to catch more customers before they leave.


In [ ]:
sampler = SMOTEENN(random_state=RANDOM_STATE)
X_train_resampled, y_train_resampled = sampler.fit_resample(X_train, y_train)

print("Before SMOTEENN:")
print(y_train.value_counts())
print("
After SMOTEENN:")
print(y_train_resampled.value_counts())


## Model Training

We train a mix of interpretable baseline and stronger non-linear models:

- Logistic Regression: strong, explainable baseline.
- Decision Tree: easy to explain but prone to overfitting.
- Random Forest: robust ensemble model.
- XGBoost: high-performing gradient boosting model commonly used on tabular data.


In [ ]:
models = {
    "Logistic Regression": Pipeline(
        steps=[
            ("scaler", StandardScaler()),
            ("model", LogisticRegression(max_iter=1000, class_weight="balanced", random_state=RANDOM_STATE)),
        ]
    ),
    "Decision Tree": DecisionTreeClassifier(
        criterion="gini",
        max_depth=6,
        min_samples_leaf=8,
        class_weight="balanced",
        random_state=RANDOM_STATE,
    ),
    "Random Forest": RandomForestClassifier(
        n_estimators=250,
        max_depth=8,
        min_samples_leaf=4,
        class_weight="balanced",
        random_state=RANDOM_STATE,
        n_jobs=-1,
    ),
    "XGBoost": XGBClassifier(
        objective="binary:logistic",
        eval_metric="logloss",
        tree_method="hist",
        n_estimators=200,
        learning_rate=0.05,
        max_depth=4,
        subsample=0.9,
        colsample_bytree=0.9,
        random_state=RANDOM_STATE,
        n_jobs=-1,
    ),
}

results = []
fitted_models = {}

for name, model in models.items():
    model.fit(X_train_resampled, y_train_resampled)
    fitted_models[name] = model
    results.append(evaluate_classifier(model, X_test, y_test, name))

comparison_df = metrics_table(results)
comparison_df


## Model Evaluation

ROC-AUC measures ranking quality across thresholds, while recall shows how many churners we catch at the default threshold. For retention, a good model should balance churn detection with practical campaign targeting.


In [ ]:
for result in results:
    print(f"
--- {result['model']} ---")
    print("Confusion matrix:")
    print(result["confusion_matrix"])
    print("
Classification report:")
    print(result["classification_report"])


In [ ]:
best_baseline_name = comparison_df.iloc[0]["model"]
best_baseline_model = fitted_models[best_baseline_name]

ConfusionMatrixDisplay.from_estimator(best_baseline_model, X_test, y_test, cmap="Blues")
plt.title(f"Confusion Matrix - {best_baseline_name}")
plt.show()


## Model Comparison

The comparison table ranks models by ROC-AUC, then recall and F1-score. In churn prediction, the final choice should also consider operational practicality: explainability, stability, and whether retention teams can act on the predictions.


In [ ]:
comparison_df.style.format({
    "accuracy": "{:.3f}",
    "precision": "{:.3f}",
    "recall": "{:.3f}",
    "f1": "{:.3f}",
    "roc_auc": "{:.3f}",
})


## Hyperparameter Tuning

We tune Random Forest and XGBoost with compact `RandomizedSearchCV` spaces. The search is intentionally moderate so the notebook remains practical on a laptop.


In [ ]:
rf_param_dist = {
    "n_estimators": randint(150, 401),
    "max_depth": [5, 8, 12, None],
    "min_samples_split": randint(2, 12),
    "min_samples_leaf": randint(1, 6),
    "class_weight": [None, "balanced"],
}

rf_search = RandomizedSearchCV(
    estimator=RandomForestClassifier(random_state=RANDOM_STATE, n_jobs=-1),
    param_distributions=rf_param_dist,
    n_iter=12,
    scoring="roc_auc",
    cv=3,
    random_state=RANDOM_STATE,
    n_jobs=-1,
    verbose=1,
)
rf_search.fit(X_train_resampled, y_train_resampled)

rf_search.best_params_, rf_search.best_score_


In [ ]:
xgb_param_dist = {
    "n_estimators": randint(100, 351),
    "max_depth": randint(3, 7),
    "learning_rate": uniform(0.03, 0.12),
    "subsample": uniform(0.75, 0.25),
    "colsample_bytree": uniform(0.75, 0.25),
    "min_child_weight": randint(1, 7),
}

xgb_search = RandomizedSearchCV(
    estimator=XGBClassifier(
        objective="binary:logistic",
        eval_metric="logloss",
        tree_method="hist",
        random_state=RANDOM_STATE,
        n_jobs=-1,
    ),
    param_distributions=xgb_param_dist,
    n_iter=12,
    scoring="roc_auc",
    cv=3,
    random_state=RANDOM_STATE,
    n_jobs=-1,
    verbose=1,
)
xgb_search.fit(X_train_resampled, y_train_resampled)

xgb_search.best_params_, xgb_search.best_score_


In [ ]:
tuned_models = {
    "Random Forest Tuned": rf_search.best_estimator_,
    "XGBoost Tuned": xgb_search.best_estimator_,
}

tuned_results = [
    evaluate_classifier(model, X_test, y_test, name)
    for name, model in tuned_models.items()
]

final_comparison_df = metrics_table(results + tuned_results)
final_comparison_df


In [ ]:
# Selection rule: highest ROC-AUC, with recall and model practicality as tie-breakers.
best_model_name = final_comparison_df.iloc[0]["model"]
best_model = {**fitted_models, **tuned_models}[best_model_name]

print(f"Selected model: {best_model_name}")
final_comparison_df.head()


## Model Explainability

Tree-based models expose feature importance, and SHAP helps show how features push predictions toward or away from churn. These explanations make the model easier to discuss with business stakeholders and interviewers.


In [ ]:
def get_feature_importance(model, feature_names):
    if isinstance(model, Pipeline):
        coefficients = model.named_steps["model"].coef_[0]
        importance = np.abs(coefficients)
    elif hasattr(model, "feature_importances_"):
        importance = model.feature_importances_
    else:
        importance = np.zeros(len(feature_names))

    return (
        pd.DataFrame({"feature": feature_names, "importance": importance})
        .sort_values("importance", ascending=False)
        .reset_index(drop=True)
    )

feature_importance_df = get_feature_importance(best_model, X.columns)
feature_importance_df.head(15)


In [ ]:
plt.figure(figsize=(10, 6))
sns.barplot(
    data=feature_importance_df.head(15),
    x="importance",
    y="feature",
    color="#2a9d8f",
)
plt.title(f"Top Churn Drivers - {best_model_name}")
plt.xlabel("Importance")
plt.ylabel("")
plt.tight_layout()
plt.show()


In [ ]:
try:
    import shap

    shap_sample = X_test.sample(min(500, len(X_test)), random_state=RANDOM_STATE)
    explainer_model = best_model.named_steps["model"] if isinstance(best_model, Pipeline) else best_model

    if isinstance(best_model, Pipeline):
        shap_input = pd.DataFrame(
            best_model.named_steps["scaler"].transform(shap_sample),
            columns=shap_sample.columns,
            index=shap_sample.index,
        )
        explainer = shap.LinearExplainer(explainer_model, shap_input)
        shap_values = explainer(shap_input)
    else:
        shap_input = shap_sample
        explainer = shap.TreeExplainer(explainer_model)
        shap_values = explainer(shap_input)

    shap.summary_plot(shap_values, shap_input, max_display=15, show=True)
except Exception as exc:
    print(f"SHAP plot skipped: {exc}")


In [ ]:
top_churn_drivers = feature_importance_df.head(10).copy()
top_churn_drivers


### Business Meaning of Important Features

Common churn drivers in this dataset usually map to clear retention actions:

- `Contract_Month-to-month`: customers without long-term commitment are more likely to churn; offer upgrade incentives or loyalty bundles.
- `tenure_group_1 - 12`: early lifecycle customers need onboarding, product education, and proactive support.
- `OnlineSecurity_No` and `TechSupport_No`: missing support/security services may signal lower perceived value; cross-sell helpful add-ons carefully.
- `InternetService_Fiber optic`: high-speed customers often pay more, so service reliability and price-value perception matter.
- `PaymentMethod_Electronic check`: this group often shows higher churn; payment friction or customer segment behavior may need investigation.


## Business Recommendations

1. High churn probability: trigger a retention campaign with a personalized offer before renewal or billing date.
2. Month-to-month contract risk: promote annual contracts with small discounts, free upgrades, or loyalty benefits.
3. Pricing sensitivity: monitor high `MonthlyCharges` customers and test targeted bundles instead of blanket discounts.
4. Customer support intervention: prioritize customers without tech support or online security for onboarding calls and service education.
5. Early-tenure customers: create a first-90-day customer success workflow to reduce avoidable churn.

The model should support retention teams, not replace business judgment. A useful deployment would rank customers weekly, explain the top churn reasons, and track whether interventions reduce churn.


## Model Saving

The app needs more than a raw estimator. We save a dictionary containing the trained model, feature columns, metrics, feature importance, and prediction threshold.


In [ ]:
MODEL_PATH.parent.mkdir(parents=True, exist_ok=True)

artifact = {
    "model": best_model,
    "model_name": best_model_name,
    "feature_columns": list(X.columns),
    "metrics": final_comparison_df.to_dict(orient="records"),
    "feature_importance": feature_importance_df.head(20).to_dict(orient="records"),
    "threshold": 0.5,
}

with open(MODEL_PATH, "wb") as file:
    pickle.dump(artifact, file)

MODEL_PATH


## Final Conclusion

This notebook now represents an end-to-end churn analytics workflow:

- Cleaned and transformed raw Telco customer data.
- Preserved original tenure-based feature engineering.
- Handled class imbalance without leaking test data.
- Compared Logistic Regression, Decision Tree, Random Forest, and XGBoost.
- Tuned Random Forest and XGBoost with practical search spaces.
- Selected the best model using ROC-AUC plus retention practicality.
- Explained churn drivers with feature importance and SHAP.
- Saved a deployment-ready artifact for the Streamlit app.
